## 基础对话

In [7]:
import sys
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from config import config

model = init_chat_model(
    "Qwen/Qwen3.5-27B",
    model_provider="openai",
    api_key=config.MODELSCOPE_API_KEY,
    base_url=config.MODELSCOPE_BASE_URL,
)

agent = create_agent(
    model=model,
    system_prompt="你是一个有帮助的助手。",
    tools=[],
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，介绍一下你自己"}]}
)

print(response["messages"][-1].content)

你好！我是 Qwen，由阿里巴巴集团研发的超大规模语言模型。我具备以下核心能力：

1. **多领域专业支持**
- 支持中文/英文等百种语言流畅交互
- 可处理数学计算、代码生成与调试（支持 Python/JavaScript/Java/CSS 等主流语言）
- 支持图表数据理解与信息抽取分析

2. **内容创作与优化**
- 能根据用户需求定制专业文案或故事创作
- 可执行逻辑推理任务，如拆解复杂问题提供解决方案
- 支持文本情感倾向判断与语义相似度匹配

3. **多场景适应能力**
- 在医疗、法律、科技等专业领域可提供标准化知识框架参考
- 支持对话式交互中的上下文精准记忆（基于超长窗口机制）
- 对图片、表格等非结构化信息具备初步解析能力

我的训练数据截止时间为 2024 年，所有功能模块均经过行业标准的合规性验证。需要任何帮助时，请随时告诉我具体需求。


## 意图识别

In [3]:
import sys
import json
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))

from enum import Enum
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from config import config

# 四类意图枚举
class Intent(str, Enum):
    incomplete_params = "incomplete_params"  # 与时薪相关但参数不完整
    complete_params = "complete_params"      # 与时薪相关且参数完整
    single_h = "single_h"                   # 单独输入字母 h
    unrelated = "unrelated"                 # 与时薪无关

class IntentResult(BaseModel):
    intent: Intent = Field(description="用户意图分类")
    reason: str = Field(description="分类理由，简短说明")

model = init_chat_model(
    "Qwen/Qwen3.5-27B",
    model_provider="openai",
    api_key=config.MODELSCOPE_API_KEY,
    base_url=config.MODELSCOPE_BASE_URL,
)

# ModelScope 只支持 json_object，用 json_mode 并在 prompt 中描述 schema
structured_model = model.with_structured_output(IntentResult, method="json_mode")

SYSTEM_PROMPT = """你是一个意图识别助手，专门分析用户输入与时薪计算的相关性。

请将用户输入分类为以下四种意图之一：

1. incomplete_params：用户输入与时薪计算有关，但参数不完整。
   时薪计算所需完整参数为：
   - 月薪
   - 每天上班时间（默认上午）
   - 每天下班时间（默认下午）
   - 每周工作天数
   以上任意一项缺失，则选择此意图。

2. complete_params：用户输入与时薪计算有关，且四项参数（月薪、上班时间、下班时间、每周工作天数）全部提供。

3. single_h：用户单独输入了字母 h（大小写均算，且不含其他内容）。

4. unrelated：用户输入与时薪计算完全无关。

必须以 JSON 格式输出，格式如下：
{"intent": "<意图值>", "reason": "<简短理由>"}

其中 intent 的值只能是以下四者之一：
incomplete_params | complete_params | single_h | unrelated"""

# 测试用例
test_cases = [
    "我月薪8000，早上9点上班，下午6点下班，时薪多少？",
    "我月薪10000，早上9点上班，下午6点下班，每周工作5天，时薪是多少？",
    "h",
    "今天天气真好",
]

for user_input in test_cases:
    result = structured_model.invoke([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input},
    ])
    print(f"输入: {user_input}")
    print(f"意图: {result.intent.value}")
    print(f"理由: {result.reason}")
    print("-" * 50)


输入: 我月薪8000，早上9点上班，下午6点下班，时薪多少？
意图: incomplete_params
理由: 输入包含了月薪和上下班时间，但缺失每周工作天数参数
--------------------------------------------------
输入: 我月薪10000，早上9点上班，下午6点下班，每周工作5天，时薪是多少？
意图: complete_params
理由: 用户提供了月薪、上班时间、下班时间及每周工作天数全部四项参数，且询问时薪。
--------------------------------------------------
输入: h
意图: single_h
理由: 用户输入仅为字母 h，无其他内容
--------------------------------------------------
输入: 今天天气真好
意图: unrelated
理由: 用户输入的是关于天气的日常寒暄，完全不涉及时薪计算相关的参数或请求。
--------------------------------------------------


## 工具调用

In [5]:
import sys
from datetime import datetime
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from config import config

# 将函数逻辑封装为 LangChain tool
@tool
def get_current_year() -> str:
    """获取当前年份，返回四位数字字符串。"""
    return str(datetime.now().year)

model = init_chat_model(
    "Qwen/Qwen3.5-27B",
    model_provider="openai",
    api_key=config.MODELSCOPE_API_KEY,
    base_url=config.MODELSCOPE_BASE_URL,
)

agent = create_agent(
    model=model,
    system_prompt="你是一个有帮助的助手，可以使用工具回答问题。",
    tools=[get_current_year],
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "你有哪些工具？"}]}
)

print(response["messages"][-1].content)


根据我的配置，我目前有1个工具可以使用：

1. **get_current_year** - 获取当前年份，返回四位数字字符串

如果您需要查询当前年份，我可以帮您调用这个工具。有什么具体问题需要我帮忙解答吗？


## 参数提取 + 时薪计算

- 假设输入的query已经是经过意图分类筛选后的，也就是包含完整参数

In [7]:
import sys
import json
import datetime
import calendar
import requests
from pathlib import Path
str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from config import config

# ---------- 参数提取的结构化输出模型 ----------

class ExtractedParams(BaseModel):
    salary: float = Field(description="月薪数字")
    work_time: int = Field(description="上班时间（24小时制）")
    off_time: int = Field(description="下班时间（24小时制）")
    week_days: int = Field(description="每周工作天数，默认5")
    month_in_prompt: str = Field(description="月份字符串，格式 2025-03，无则为 '0'")

# ---------- 时薪计算器工具 ----------

def _get_month_work_days(year: int, month: int, week_days: int) -> tuple:
    """查询节假日 API，返回 (total, work, rest, rest_list, work_list)。"""
    total_days = calendar.monthrange(year, month)[1]
    work_days, rest_days = 0, 0
    rest_day_list, work_day_list = [], []

    try:
        url = f"http://timor.tech/api/holiday/year/{year}-{month:02d}"
        headers = {"User-Agent": "Mozilla/5.0", "Referer": "http://timor.tech/"}
        resp = requests.get(url, headers=headers, timeout=5)
        resp.raise_for_status()
        holiday_info = resp.json()

        for day in range(1, total_days + 1):
            date = datetime.date(year, month, day)
            day_str = f"{month:02d}-{day:02d}"
            weekday = date.weekday()
            if day_str in holiday_info.get("holiday", {}):
                if holiday_info["holiday"][day_str].get("holiday", False):
                    rest_days += 1
                    rest_day_list.append(date.strftime("%Y-%m-%d"))
                else:  # 补班
                    work_days += 1
                    work_day_list.append(date.strftime("%Y-%m-%d"))
            else:
                if weekday < week_days:
                    work_days += 1
                    work_day_list.append(date.strftime("%Y-%m-%d"))
                else:
                    rest_days += 1
                    rest_day_list.append(date.strftime("%Y-%m-%d"))
    except Exception:
        # API 不可用时退回纯日历计算
        for day in range(1, total_days + 1):
            date = datetime.date(year, month, day)
            if date.weekday() < week_days:
                work_days += 1
                work_day_list.append(date.strftime("%Y-%m-%d"))
            else:
                rest_days += 1
                rest_day_list.append(date.strftime("%Y-%m-%d"))

    return total_days, work_days, rest_days, rest_day_list, work_day_list


@tool
def calculate_hourly_rate(
    salary: float,
    work_time: float,
    off_time: float,
    week_days: int,
    month: str,
) -> str:
    """计算时薪及月度工作信息。
    salary: 月薪；work_time: 上班时间（24小时制）；off_time: 下班时间（24小时制）；
    week_days: 每周工作天数；month: 月份字符串 2025-03，无则传 '0'。
    """
    daily_work_time = off_time - work_time
    if daily_work_time < 0:
        daily_work_time += 24

    if month == "0":
        now = datetime.datetime.now()
        year_num, month_num = now.year, now.month
        month_str = f"{year_num}-{month_num:02d}"
    else:
        year_num, month_num = map(int, month.split("-"))
        month_str = month

    total_days, work_days, rest_days, rest_day_list, work_day_list = (
        _get_month_work_days(year_num, month_num, week_days)
    )
    hourly_rate = salary / (work_days * daily_work_time)

    return json.dumps({
        "month_str": month_str,
        "daily_work_time": daily_work_time,
        "monthly_work_time": daily_work_time * work_days,
        "hourly_rate": round(hourly_rate, 2),
        "total_days": total_days,
        "work_days": work_days,
        "rest_days": rest_days,
        "rest_day_list": rest_day_list,
        "work_day_list": work_day_list,
    }, ensure_ascii=False)


# ---------- 参数提取 ----------

current_year = datetime.datetime.now().year

model = init_chat_model(
    "Qwen/Qwen3.5-27B",
    model_provider="openai",
    api_key=config.MODELSCOPE_API_KEY,
    base_url=config.MODELSCOPE_BASE_URL,
)

param_extractor = model.with_structured_output(ExtractedParams, method="json_mode")

EXTRACT_SYSTEM_PROMPT = f"""用户输入包含：月薪、周作息【每天上班时间（默认上午）、每天下班时间（默认下午）、每周工作天数】的数字组合或自然语言描述
### 工作时间输入参考：
- 955（每天早上9点上班，每天下午5点下班，每周工作5天）
- 996（每天早上9点上班，每天下午9点下班，每周工作6天）
- 965（每天早上9点上班，每天下午6点下班，每周工作5天）
- 966（每天早上9点上班，每天下午6点下班，每周工作6天）
- 10107（每天早上10点上班，每天下午10点下班，每周工作7天）
- 其他的组合你根据数字的实际情况拆分理解输出正确的参数！
## 提取参数
### salary[number]
从用户的输入中提取月薪的具体数字，如果是 10k 则转成 10000。
### work_time[number]
提取上班时间，输出 24 小时制数字。
### off_time[number]
提取下班时间，输出 24 小时制数字。
案例：周作息 965，第二位 6 是下班时间，24 小时制 6+12=18，输出 18。
### week_days[number]
提取每周工作天数，未检测到则默认 5。
### month_in_prompt[string]
判断是否有明确月份：有则输出 2025-03 格式；无则输出字符串 0；
只有月份时拼接当前年份；有年月则直接输出。

当前实际年份：{current_year}

必须以 JSON 格式输出，字段：salary, work_time, off_time, week_days, month_in_prompt"""

user_input = "我月薪15000，965，帮我算一下时薪"

params: ExtractedParams = param_extractor.invoke([
    {"role": "system", "content": EXTRACT_SYSTEM_PROMPT},
    {"role": "user", "content": user_input},
])

print("提取的参数：")
print(f"  月薪:       {params.salary}")
print(f"  上班时间:   {params.work_time}:00")
print(f"  下班时间:   {params.off_time}:00")
print(f"  每周工作:   {params.week_days} 天")
print(f"  月份:       {params.month_in_prompt}")
print()

# ---------- 调用时薪计算工具 ----------

result_str = calculate_hourly_rate.invoke({
    "salary": params.salary,
    "work_time": params.work_time,
    "off_time": params.off_time,
    "week_days": params.week_days,
    "month": params.month_in_prompt,
})

result = json.loads(result_str)

# 将所有变量解包到顶层，供后续 cell 直接使用
salary           = params.salary
work_time        = params.work_time
off_time         = params.off_time
week_days        = params.week_days
month_str        = result["month_str"]
total_days       = result["total_days"]
rest_days        = result["rest_days"]
work_days        = result["work_days"]
daily_work_time  = result["daily_work_time"]
monthly_work_time = result["monthly_work_time"]
hourly_rate      = result["hourly_rate"]
rest_day_list    = result["rest_day_list"]
work_day_list    = result["work_day_list"]

print(f"月份:         {month_str}")
print(f"每日工时:     {daily_work_time} 小时")
print(f"月度总工时:   {monthly_work_time} 小时")
print(f"工作天数:     {work_days} 天（休息 {rest_days} 天）")
print(f"时薪:         ¥{hourly_rate:.2f}")


提取的参数：
  月薪:       15000.0
  上班时间:   9:00
  下班时间:   18:00
  每周工作:   5 天
  月份:       0

月份:         2026-03
每日工时:     9.0 小时
月度总工时:   198.0 小时
工作天数:     22 天（休息 9 天）
时薪:         ¥75.76


## 总结输出

In [8]:
from IPython.display import Markdown, display

SUMMARY_PROMPT = f"""以 markdown 的格式总结用户的时薪计算情况（不要使用代码块包裹，直接输出即可）

1. 以 markdown 表格的形式直观展示用户的数据：
月薪：{salary}
每天上班时间：{work_time} 点
每天下班时间：{off_time} 点
每周工作天数：{week_days}
月份：{month_str}
该月天数：{total_days}
该月休息日天数：{rest_days}
该月工作日天数：{work_days}
每天工作时长：{daily_work_time}
该月具体工作时长（小时）：{monthly_work_time}
时薪：{hourly_rate}

2. 使用 LaTeX 公式详细展示时薪计算的过程

3. 展示当月休息的日子：{rest_day_list}"""

resp = model.invoke([{"role": "user", "content": SUMMARY_PROMPT}])
display(Markdown(resp.content))


### 时薪计算详情表

| 项目                 | 数值/说明           |
|----------------------|--------------------|
| **月薪**             | ¥15,000.0          |
| **每日工作时间段**   | 9:00 - 18:00       |
| **周工作频率**       | 5 天               |
| **统计月份**          | 2026-03            |
| **当月总天数**        | 31 天              |
| **当月休息日天数**    | 9 天                |
| **当月工作日天数**    | 22 天               |
| **日均工作时长**      | 9.0 小时            |
| **当月总工作时长**    | 198.0 小时           |
| **计算出结果（时薪）** | ¥75.76/小时         |

---

### 时薪计算过程
根据已知条件，时薪计算公式为：
$$ \text{时薪} = \frac{\text{月薪}}{\text{月总工作时长}} $$
代入实际数据：
$$ \text{时薪} = \frac{15000}{22 \times 9} = \frac{15000}{198} \approx 75.76 $$

---

### 当月休息日
- 2026-03-01
- 2026-03-07
- 2026-03-08
- 2026-03-14
- 2026-03-15
- 2026-03-21
- 2026-03-22
- 2026-03-28
- 2026-03-29